In [56]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error

In [57]:
df = pd.read_excel(r"C:\Users\yapen\Desktop\TrainDataset2025.xls")

In [58]:
df = df.drop(columns=["ID"])
df = df.drop(columns=['pCR (outcome)'])

In [59]:
df = df.replace(999, np.nan)

In [60]:
df["HistologyType"] = df["HistologyType"].map({1:0, 2:1})
ordinal_cols = ["TumourStage", "Proliferation", "ChemoGrade"]
df[ordinal_cols] = df[ordinal_cols].astype("Int64")

In [61]:
mask = df.isna().sum(axis=1) > 1
df = df.drop(df[mask].index)

In [62]:
df = df.dropna(subset=['LNStatus'])

In [63]:
X = df.drop(columns=['RelapseFreeSurvival (outcome)'])

y = df['RelapseFreeSurvival (outcome)']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

In [64]:
gene_col = "Gene"
# -------- TRAIN SET --------
mask_train_known   = X_train[gene_col].notna() # true where gene present
mask_train_missing = X_train[gene_col].isna() # true where gene is NaN

# numeric predictors excluding gene
feat_cols = X_train.select_dtypes(include="number").columns.drop(gene_col)

X_known = X_train.loc[mask_train_known, feat_cols]
y_known = X_train.loc[mask_train_known, gene_col]
X_missing_train = X_train.loc[mask_train_missing, feat_cols]

# Train classifier on TRAIN ONLY
clf = RandomForestClassifier(random_state=0)
clf.fit(X_known, y_known)

joblib.dump(clf, "gene_classifier.pkl")
print("Saved gene classifier.")

# Fill missing Gene IN TRAIN
pred_gene_train = clf.predict(X_missing_train)
X_train.loc[mask_train_missing, gene_col] = pred_gene_train


# -------- TEST SET: apply classifier (NO FITTING) --------
mask_test_missing = X_test[gene_col].isna()
X_missing_test = X_test.loc[mask_test_missing, feat_cols]

if mask_test_missing.sum() > 0:
    pred_gene_test = clf.predict(X_missing_test)
    X_test.loc[mask_test_missing, gene_col] = pred_gene_test

Saved gene classifier.


In [65]:
X_train_df = pd.DataFrame(X_train, columns=X_train.columns)
X_test_df  = pd.DataFrame(X_test,  columns=X_test.columns)


Q1 = X_train_df.quantile(0.25)
Q3 = X_train_df.quantile(0.75)
IQR = Q3 - Q1

outlier_info = {
    "Q1": Q1,
    "Q3": Q3
}
joblib.dump(outlier_info, "outlier_params.pkl")
print("Saved outlier parameters.")


X_train_clip = X_train_df.clip(
    lower=Q1 - 1.5 * IQR,
    upper=Q3 + 1.5 * IQR,
    axis=1
)

X_test_clip = X_test_df.clip(
    lower=Q1 - 1.5 * IQR,
    upper=Q3 + 1.5 * IQR,
    axis=1
)

Saved outlier parameters.


In [66]:
# 1. Define the model
model = RandomForestRegressor(
    n_estimators=38,
    max_depth=2,
    min_samples_split=5,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=True,
    max_samples=None,
    n_jobs=-1,
    random_state=42
)

# 2. 5-fold cross-validation (main evaluation step)
cv_scores = -cross_val_score(
    model,
    X_train_clip,
    y_train,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

print(f"CV MAE (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# 3. Train the final model on all training data
model.fit(X_train_clip, y_train)

joblib.dump(model, "rfs_rf_model.pkl")
print("Saved RFS RF model.")


# 4. Predict on train and test
y_train_pred = model.predict(X_train_clip)
y_test_pred  = model.predict(X_test_clip)

print(f"Train MAE: {mean_absolute_error(y_train, y_train_pred):.4f}")
print(f"Test MAE:  {mean_absolute_error(y_test,  y_test_pred):.4f}")


CV MAE (5-fold): 20.6098 ± 2.2561
Saved RFS RF model.
Train MAE: 19.1895
Test MAE:  21.8409
